# EA1 - Actividad 1.2: RDDs (Resilient Distributed Datasets)

## Objetivos
- Entender que son los RDDs y sus propiedades fundamentales
- Diferenciar entre transformaciones (lazy) y acciones
- Aplicar operaciones basicas: map, filter, flatMap, reduce, reduceByKey
- Comparar el rendimiento de RDDs vs DataFrames

## Conceptos Clave

### Que son los RDDs?

Los **RDD (Resilient Distributed Datasets)** son la estructura de datos fundamental de Spark. Son una coleccion distribuida e inmutable de elementos que se pueden procesar en paralelo.

**Propiedades principales:**

| Propiedad | Descripcion |
|-----------|-------------|
| **Inmutabilidad** | Una vez creado, un RDD no se puede modificar. Cada transformacion crea un nuevo RDD |
| **Distribuido** | Los datos se dividen en **particiones** que se distribuyen entre los nodos del cluster |
| **Resiliente** | Si una particion se pierde, Spark puede reconstruirla gracias al **lineage** (historial de transformaciones) |
| **Evaluacion lazy** | Las transformaciones no se ejecutan inmediatamente; se acumulan y se ejecutan cuando se invoca una accion |

### Transformaciones vs Acciones

- **Transformaciones:** Crean un nuevo RDD a partir de uno existente. Son **lazy** (no se ejecutan de inmediato).
  - Ejemplos: `map()`, `filter()`, `flatMap()`, `reduceByKey()`, `groupByKey()`

- **Acciones:** Devuelven un resultado al driver o escriben datos. **Disparan la ejecucion** de todas las transformaciones pendientes.
  - Ejemplos: `collect()`, `count()`, `take()`, `reduce()`, `first()`, `saveAsTextFile()`

In [ ]:
# Setup: Crear SparkSession y obtener SparkContext
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("02_rdds_basico") \
    .master("local[*]") \
    .getOrCreate()

# SparkContext es el punto de entrada para trabajar con RDDs
sc = spark.sparkContext

print(f"Spark version: {spark.version}")
print(f"SparkContext: {sc}")

## Crear RDDs

Hay dos formas principales de crear un RDD:
1. **Desde una coleccion en memoria** con `sc.parallelize()`
2. **Desde un archivo externo** con `sc.textFile()`

In [ ]:
# Crear un RDD desde una lista de Python
rdd = sc.parallelize([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

print(f"Tipo: {type(rdd)}")
print(f"Numero de particiones: {rdd.getNumPartitions()}")

## Transformaciones (Lazy)

Las transformaciones crean un nuevo RDD pero **no se ejecutan** hasta que se invoque una accion.

### `map()` - Aplica una funcion a cada elemento

In [ ]:
# map: multiplicar cada elemento por 2
rdd_doble = rdd.map(lambda x: x * 2)

# Nota: hasta aqui NO se ha ejecutado nada!
# Solo se ha registrado la transformacion
print(f"rdd_doble es de tipo: {type(rdd_doble)}")
print("La transformacion aun no se ejecuto (lazy evaluation)")

### `filter()` - Selecciona elementos que cumplen una condicion

In [ ]:
# filter: obtener solo numeros mayores a 5
rdd_mayores = rdd.filter(lambda x: x > 5)

# Encadenar transformaciones: multiplicar por 2 Y luego filtrar > 10
rdd_encadenado = rdd.map(lambda x: x * 2).filter(lambda x: x > 10)

print("Transformaciones registradas (aun no ejecutadas)")

## Acciones (Disparan la ejecucion)

Las acciones fuerzan la evaluacion de todas las transformaciones pendientes.

In [ ]:
# collect(): devuelve TODOS los elementos como lista de Python
# CUIDADO: no usar con datasets grandes (todo va a memoria del driver)
print(f"Original: {rdd.collect()}")
print(f"Doble: {rdd_doble.collect()}")
print(f"Mayores a 5: {rdd_mayores.collect()}")
print(f"Encadenado (x2, >10): {rdd_encadenado.collect()}")

In [ ]:
# count(): cuenta el numero de elementos
print(f"Cantidad de elementos: {rdd.count()}")

# take(n): devuelve los primeros n elementos
print(f"Primeros 3: {rdd.take(3)}")

# reduce(): combina todos los elementos usando una funcion
suma = rdd.reduce(lambda a, b: a + b)
print(f"Suma total: {suma}")

## `flatMap()` - Aplanar resultados

A diferencia de `map()`, `flatMap()` puede devolver 0 o mas elementos por cada elemento de entrada, y los "aplana" en un solo RDD.

In [ ]:
# Ejemplo: dividir frases en palabras
frases = sc.parallelize([
    "Hola mundo de Spark",
    "Big Data es fascinante",
    "Spark procesa datos en paralelo"
])

# Con map: cada elemento se convierte en una lista de palabras
palabras_map = frases.map(lambda frase: frase.split(" "))
print(f"Con map (listas anidadas): {palabras_map.collect()}")

# Con flatMap: las listas se aplanan en elementos individuales
palabras_flat = frases.flatMap(lambda frase: frase.split(" "))
print(f"Con flatMap (plano): {palabras_flat.collect()}")

## Leer un archivo como RDD de texto

Con `sc.textFile()` cada linea del archivo se convierte en un elemento del RDD.

In [ ]:
# Leer flights.csv como RDD de lineas de texto
rdd_flights = sc.textFile("/home/jovyan/datos/flights.csv")

# Ver las primeras 3 lineas
for linea in rdd_flights.take(3):
    print(linea)

print(f"\nTotal de lineas (incluye header): {rdd_flights.count()}")

In [ ]:
# Obtener el header y separarlo de los datos
header = rdd_flights.first()
print(f"Header: {header}")

# Filtrar el header para quedarnos solo con los datos
rdd_datos = rdd_flights.filter(lambda linea: linea != header)
print(f"Lineas de datos: {rdd_datos.count()}")

---
## Ejercicios

Ahora es tu turno de practicar con RDDs.

In [ ]:
# =============================================================
# EJERCICIO 1: Cuadrados de numeros pares
# =============================================================
# Filtrar pares y calcular su cuadrado

rdd_resultado = rdd.filter(lambda x: x % 2 == 0).map(lambda x: x ** 2)
print(rdd_resultado.collect())

> **Nota docente:** Este ejercicio refuerza el encadenamiento de transformaciones. El orden importa: primero `filter` (reduce la cantidad de datos) y luego `map` (transforma). Esto es mas eficiente que hacer `map` primero y luego `filter`, ya que procesamos menos elementos. Error comun: los estudiantes invierten el orden o intentan hacer ambas operaciones en un solo `map` con condicional. Recordar que hasta el `collect()` no se ejecuta nada (lazy evaluation).

In [ ]:
# =============================================================
# EJERCICIO 2: Word Count (conteo de palabras)
# =============================================================
# Implementacion del clasico Word Count con RDDs

frases = sc.parallelize([
    "apache spark es rapido",
    "spark procesa datos grandes",
    "big data con spark es genial",
    "datos grandes necesitan spark"
])

conteo = frases.flatMap(lambda linea: linea.split(" ")) \
    .map(lambda palabra: (palabra, 1)) \
    .reduceByKey(lambda a, b: a + b) \
    .sortBy(lambda x: x[1], ascending=False)

print(conteo.collect())

> **Nota docente:** Word Count es el "Hello World" de Big Data. El patron `flatMap -> map -> reduceByKey` es fundamental. Puntos clave para explicar: (1) `flatMap` vs `map`: flatMap "aplana" las listas resultantes en elementos individuales; (2) `reduceByKey` es preferible a `groupByKey` porque reduce localmente antes de transferir datos entre nodos (shuffle), lo que es mucho mas eficiente; (3) `sortBy` ordena el resultado final. Error comun: usar `groupByKey` y luego sumar, lo cual es correcto pero ineficiente ya que transfiere todos los valores por la red antes de agregar.

In [ ]:
# =============================================================
# EJERCICIO 3: Vuelos con retraso mayor a 30 minutos
# =============================================================
# Parsear el CSV como RDD y filtrar vuelos con retraso > 30 min

# Identificar la posicion de dep_delay en el header
header = rdd_flights.first()
print(f"Header: {header}")
columnas = header.split(",")
print(f"Indice de dep_delay: {columnas.index('dep_delay')}")

# Filtrar header, parsear y filtrar por retraso > 30
# dep_delay es StringType (puede contener "NA"), indice 4 en el schema real
rdd_data = rdd_flights.filter(lambda linea: linea != header)
rdd_parsed = rdd_data.map(lambda linea: linea.split(","))

# dep_delay esta en el indice 4 (verificar con el header)
rdd_retrasados = rdd_parsed.filter(
    lambda campos: campos[4] != "" and campos[4] != "NA" and float(campos[4]) > 30
)
print(f"Vuelos con retraso > 30 min: {rdd_retrasados.count()}")

> **Nota docente:** Este ejercicio es deliberadamente mas dificil porque trabaja con datos reales en formato texto. Los errores mas comunes son: (1) olvidar remover el header antes de procesar; (2) no manejar valores vacios o "NA" antes de convertir a float (genera excepciones) — en este dataset `dep_delay` es StringType y contiene "NA" para vuelos sin datos; (3) usar el indice incorrecto para la columna. Es buena practica mostrar a los estudiantes como verificar el indice programaticamente con `columnas.index('dep_delay')`. Comparar con la facilidad que ofrece un DataFrame para esta misma tarea: `df.withColumn("dep_delay", F.col("dep_delay").cast("double")).filter(F.col("dep_delay") > 30).count()` -- esto motiva el uso de DataFrames.

---
## Resumen

En esta actividad aprendimos:

1. **RDDs:** Estructura de datos fundamental de Spark — inmutable, distribuida, resiliente
2. **Transformaciones (lazy):** `map()`, `filter()`, `flatMap()` — no se ejecutan hasta que hay una accion
3. **Acciones:** `collect()`, `count()`, `take()`, `reduce()` — disparan la ejecucion
4. **Crear RDDs:** Desde listas (`sc.parallelize()`) o archivos (`sc.textFile()`)
5. **Word Count:** Patron clasico de Big Data usando `flatMap` + `map` + `reduceByKey`
6. **Parseo de archivos CSV:** Separar header, split por coma, conversion de tipos

---
## Desafio Extra (Opcional)

**Comparar rendimiento: RDD vs DataFrame**

Realiza la misma operacion (contar vuelos por aerolinea) usando RDD y DataFrame.
Usa `%%time` al inicio de cada celda para comparar el tiempo de ejecucion.

Pregunta: Cual es mas rapido? Por que crees que ocurre esa diferencia?
(Pista: investiga el optimizador Catalyst de Spark SQL)

In [ ]:
# =============================================================
# DESAFIO: Comparar RDD vs DataFrame
# =============================================================
# Medir tiempos de ejecucion para la misma operacion

import time

# --- Parte A: Con RDD ---
start = time.time()
header = rdd_flights.first()
# carrier esta en el indice 7: year(0),month(1),day(2),dep_time(3),dep_delay(4),
#   arr_time(5),arr_delay(6),carrier(7)
conteo_rdd = rdd_flights.filter(lambda l: l != header) \
    .map(lambda l: l.split(",")) \
    .map(lambda c: (c[7], 1)) \
    .reduceByKey(lambda a, b: a + b) \
    .collect()
tiempo_rdd = time.time() - start

# --- Parte B: Con DataFrame ---
df = spark.read.csv("/home/jovyan/datos/flights.csv", header=True, inferSchema=True)

start = time.time()
conteo_df = df.groupBy("carrier").count().collect()
tiempo_df = time.time() - start

print(f"RDD: {tiempo_rdd:.2f}s")
print(f"DataFrame: {tiempo_df:.2f}s")
print(f"DataFrame es {tiempo_rdd/tiempo_df:.1f}x mas rapido")

> **Nota docente:** El DataFrame es significativamente mas rapido gracias al optimizador **Catalyst** y al motor de ejecucion **Tungsten**. Catalyst analiza la consulta completa y genera un plan de ejecucion optimizado (similar a un query planner de SQL), mientras que Tungsten gestiona la memoria de forma eficiente con serializacion binaria en lugar de objetos Java/Python. Con RDDs, Spark no puede optimizar porque las lambdas son opacas (no sabe que hacen). Ademas, los RDDs con Python sufren la sobrecarga de serializar datos entre la JVM y el proceso Python. Este es el argumento principal para preferir DataFrames en la mayoria de los casos.

In [ ]:
# Detener la SparkSession al finalizar
spark.stop()
print("SparkSession detenida correctamente.")